In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── CẤU HÌNH ──────────────────────────────────────────────────────
DRIVE_OUT   = "/content/drive/MyDrive/vimfd/outputs"               # nơi lưu kết quả (checkpoint, csv, log)
RUN_V3_V4   = False      # False: dùng product_attrs.json / pseudo_*.csv đã chạy ở máy bạn (khuyên dùng)
STRICT_N1   = False      # True nếu kiểm tra 20 cặp âm ở V5 được < 18 (chạy lại V5 với cờ này)
V6_SMOKE    = True       # chạy thử luồng V6 trước (~vài phút) rồi mới chạy thật
V6_TRAIN_FRAC = 1.0      # 0.5 nếu sợ không kịp thời gian

In [ ]:
import os, zipfile, shutil
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/thesis/vimfd-cross-encoder.zip"
FORCE_REEXTRACT = True

# Tìm tên thư mục gốc nằm trong file zip trước
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    first_item = z.namelist()[0]
    root_dir_name = first_item.split('/')[0]

ROOT = Path(f"/content/{root_dir_name}")
print(f"Thư mục gốc phát hiện được: {ROOT}")

if FORCE_REEXTRACT and ROOT.exists():
    shutil.rmtree(ROOT)

if not ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content/')

os.chdir(ROOT)
print("Current Directory:", os.getcwd())
print("Directory Contents:", os.listdir('.'))

Thư mục gốc phát hiện được: /content/vimfd-retrieval
Current Directory: /content/vimfd-retrieval
Directory Contents: ['config.py', 'src', 'data', 'checkpoints']


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "KHÔNG có GPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"

GPU 0: Tesla T4 (UUID: GPU-60cb775d-4980-a297-b7dc-3f3a0f9284d7)



In [ ]:
!pip -q install faiss-cpu openpyxl scikit-learn

## Kiểm tra môi trường (bắt buộc đạt trước khi chạy V5)

In [ ]:
import sys, json, os
from pathlib import Path
sys.path.insert(0, os.getcwd())
ROOT = Path(os.getcwd())

import config
print("config.CHECKPOINT_BEST =", getattr(config, "CHECKPOINT_BEST", None))
print("config.PROCESSED_DIR   =", getattr(config, "PROCESSED_DIR", None))
ck = os.environ.get("VIMFD_DE_CKPT") or getattr(config, "CHECKPOINT_BEST", None) or "checkpoints/best_24112026.pt"
ck = Path(ck) if Path(ck).is_absolute() else ROOT / ck
print("checkpoint tồn tại:", ck.exists(), ck)

for f in ["data/splits/train.jsonl", "data/splits/val.jsonl",
          "data/annotation/cross_encoder/product_attrs.json",
          "data/annotation/cross_encoder/attr_dictionary.xlsx",
          "data/annotation/cross_encoder/pseudo_train.csv",
          "data/annotation/cross_encoder/pseudo_val.csv"]:
    print(("OK   " if (ROOT / f).exists() else "THIẾU"), f)

# ảnh: thử 300 sản phẩm Train theo đúng quy tắc của V5
def featured(p):
    ims = sorted(p.get("images") or [], key=lambda x: (not x.get("is_featured", False), x.get("position", 9999)))
    if not ims or not ims[0].get("local_path"): return None
    lp = Path(ims[0]["local_path"])
    cands = [lp, ROOT / lp]
    if getattr(config, "PROCESSED_DIR", None) and not lp.is_absolute():
        cands.append(Path(config.PROCESSED_DIR) / lp)
    return next((c for c in cands if c.exists()), None)

rows = [json.loads(l) for _, l in zip(range(300), open("data/splits/train.jsonl", encoding="utf-8"))]
found = sum(featured(p) is not None for p in rows)
print(f"ảnh tìm thấy: {found}/{len(rows)}")
assert found > 0.95 * len(rows), "Không thấy ảnh: kiểm tra config.PROCESSED_DIR / vị trí thư mục ảnh trong zip"

config.CHECKPOINT_BEST = /content/vimfd-retrieval/checkpoints/best_short_24112026.pt
config.PROCESSED_DIR   = /content/vimfd-retrieval/data/processed
checkpoint tồn tại: True /content/vimfd-retrieval/checkpoints/best_short_24112026.pt
OK    data/splits/train.jsonl
OK    data/splits/val.jsonl
OK    data/annotation/cross_encoder/product_attrs.json
OK    data/annotation/cross_encoder/attr_dictionary.xlsx
OK    data/annotation/cross_encoder/pseudo_train.csv
OK    data/annotation/cross_encoder/pseudo_val.csv
ảnh tìm thấy: 300/300


## V3 và V4 (tùy chọn, mặc định bỏ qua)

In [ ]:
if RUN_V3_V4:
    !apt-get -qq install -y default-jre-headless > /dev/null
    !pip -q install py_vncorenlp
    import py_vncorenlp, os
    os.makedirs("/content/vncorenlp", exist_ok=True)
    py_vncorenlp.download_model(save_dir="/content/vncorenlp")
    os.environ["VNCORENLP_DIR"] = "/content/vncorenlp"
    !python data/annotation/v3_attributes.py
    print("\n>>> Dừng ở đây để B duyệt sheet 'synonyms' trong attr_dictionary.xlsx rồi mới chạy V4 <<<")
else:
    print("Bỏ qua V3/V4: dùng product_attrs.json và pseudo_*.csv trong zip.")

In [ ]:
if RUN_V3_V4:
    !python data/annotation/v4_pseudo_queries.py   # chỉ chạy sau khi đã duyệt bảng đồng nghĩa

## V5: khai thác âm khó (GPU)
Chạy thử luồng bằng `--dry-run` (TF-IDF, không dùng mô hình) trước, rồi chạy thật.

In [ ]:
!python data/annotation/v5_hard_negatives.py --dry-run 2>&1 | tail -15

20. [OK ] q='áo khoác bomber vintage dáng rộng' attrs={"PHONG_CACH": "vintage", "KIEU_DANG": "oversize"}
      âm: Áo khoác nam, áo khoác bomber, kiểu dáng trẻ trung thời trang, sự kết  | khớp 0/2 thuộc tính | cùng loại: True

  Tự động: 20/20 (ĐẠT).
  LƯU Ý: kết quả tự động ở trên chỉ chứng minh bộ lọc chạy đúng. Hãy đọc 20 cặp trên: ít nhất 18 cặp phải thật sự KHÔNG khớp thuộc tính truy vấn (nhìn cả name sản phẩm). Nếu <18: --strict-n1.
Đã lưu: /content/vimfd-retrieval/data/annotation/cross_encoder/ce_pairs_train_DRYRUN.csv  (25698 dòng)
Đã lưu: /content/vimfd-retrieval/data/annotation/cross_encoder/ce_pairs_val_DRYRUN.csv  (4300 dòng)

## V5 — khai thác âm khó (DRY RUN – không dùng)
- Tham số: seed=42, top_k=50, n_neg=4, strict_n1=False, thời gian=12s
- train: 5155 dương, 20543 âm (1:3.99); bỏ vì 0 âm: 19; thiếu ảnh: 0; TB bị loại/truy vấn: 9.3/50; âm cùng loại: 56%; âm cùng loại và khớp n−1 thuộc tính: 45%
- val: 860 dương, 3440 âm (1:4.00); bỏ vì 0 âm: 0; thiếu ảnh: 0; TB bị loại/

In [ ]:
flag = "--strict-n1" if STRICT_N1 else ""
# Khai báo biến môi trường chỉ định đường dẫn checkpoint chính xác trước khi chạy file python
!python -u data/annotation/v5_hard_negatives.py $flag

Load dữ liệu...
  Train 11571 | Val 1450 | pseudo_train 5174 | pseudo_val 860
  Đồng nghĩa: 25 cặp | màu thô đã map: 108
  Device: cuda
config.json: 100% 4.19k/4.19k [00:00<00:00, 12.7MB/s]

pytorch_model.bin: downloading bytes:  14% 87.7M/605M [00:01<00:06, 83.5MB/s, 5.34MB/s  ]
pytorch_model.bin: downloading bytes:  20% 123M/605M [00:02<00:04, 109MB/s, 9.81MB/s  ]  
pytorch_model.bin: reconstructing file:  17% 102M/605M [00:02<00:09, 51.4MB/s, 6.98MB/s  ]
pytorch_model.bin: downloading bytes:  26% 155M/605M [00:02<00:03, 123MB/s, 13.2MB/s  ]
pytorch_model.bin: downloading bytes:  29% 175M/605M [00:02<00:03, 135MB/s, 14.3MB/s  ]
pytorch_model.bin: downloading bytes:  34% 206M/605M [00:02<00:02, 140MB/s, 17.7MB/s  ]
pytorch_model.bin: reconstructing file:  43% 262M/605M [00:02<00:01, 191MB/s, 19.1MB/s  ]
pytorch_model.bin: downloading bytes:  41% 250M/605M [00:02<00:02, 142MB/s, 20.8MB/s  ]
pytorch_model.bin: downloading bytes:  46% 278M/605M [00:03<00:02, 140MB/s, 23.2MB/s  ]
pytorch_

**Dừng lại và ĐỌC 20 cặp âm in ở trên.** Ít nhất 18 cặp phải thật sự không khớp thuộc tính truy vấn.
Nếu dưới 18: đặt `STRICT_N1 = True` rồi chạy lại ô V5. Dòng "Tự động: x/20" luôn đạt, không thay cho việc đọc bằng mắt.

In [ ]:
# lưu kết quả V5 lên Drive (phòng khi Colab ngắt)
!mkdir -p "$DRIVE_OUT"
!cp -r data/annotation/cross_encoder/ce_pairs_*.csv "$DRIVE_OUT"/
!cp -r results/logs "$DRIVE_OUT"/logs_v5
!ls -la data/annotation/cross_encoder/ | grep ce_pairs

-rw-r--r-- 1 root root 16187506 Nov 24 16:19 ce_pairs_train.csv
-rw-r--r-- 1 root root 12313751 Nov 24 16:17 ce_pairs_train_DRYRUN.csv
-rw-r--r-- 1 root root  2613804 Nov 24 16:19 ce_pairs_val.csv
-rw-r--r-- 1 root root  2380232 Nov 24 16:17 ce_pairs_val_DRYRUN.csv


## V6: huấn luyện Cross-Encoder (GPU)
Checkpoint được ghi thẳng vào Drive (liên kết thư mục) để không mất nếu Colab ngắt giữa chừng.
Giữ tab Colab mở trong suốt thời gian chạy.

In [ ]:
import os
os.makedirs(f"{DRIVE_OUT}/cross_encoder_best", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/cross_encoder_best_smoke", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
!ln -sfn "$DRIVE_OUT/cross_encoder_best" checkpoints/cross_encoder_best
!ln -sfn "$DRIVE_OUT/cross_encoder_best_smoke" checkpoints/cross_encoder_best_smoke
!ls -la checkpoints

total 871212
drwxr-xr-x 2 root root      4096 Nov 24 16:36 .
drwxr-xr-x 8 root root      4096 Nov 24 16:17 ..
-rw-r--r-- 1 root root 892102101 Nov 24 16:15 best_short_24112026.pt
lrwxrwxrwx 1 root root        55 Nov 24 16:36 cross_encoder_best -> /content/drive/MyDrive/vimfd/outputs/cross_encoder_best
lrwxrwxrwx 1 root root        61 Nov 24 16:36 cross_encoder_best_smoke -> /content/drive/MyDrive/vimfd/outputs/cross_encoder_best_smoke


In [ ]:
if V6_SMOKE:
    !python -u data/annotation/v6_train_cross_encoder.py --smoke

[16:36:38] === Bắt đầu V6 (SMOKE TEST) ===
[16:36:38] Model: vinai/phobert-base, MAX_LEN=128, BATCH=32
[16:36:38] LR_phobert=2e-05, LR_linear=0.0001, WD=0.01, lịch=warmup 10% + cosine
[16:36:38] MAX_EPOCHS=1, PATIENCE=3, trần thời gian=2.25h
[16:36:38] FP16=True, seed=42, device=cuda
[16:36:38] GPU: Tesla T4, VRAM 15.6 GB
[16:36:39] Train: 200 cặp, Val: 100 cặp
[16:36:40] Tách token (một lần)...
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with 

In [ ]:
!python -u data/annotation/v6_train_cross_encoder.py --train-frac $V6_TRAIN_FRAC

[16:36:57] === Bắt đầu V6 ===
[16:36:57] Model: vinai/phobert-base, MAX_LEN=256, BATCH=32
[16:36:57] LR_phobert=2e-05, LR_linear=0.0001, WD=0.01, lịch=warmup 10% + cosine
[16:36:57] MAX_EPOCHS=10, PATIENCE=3, trần thời gian=2.25h
[16:36:57] FP16=True, seed=42, device=cuda
[16:36:57] GPU: Tesla T4, VRAM 15.6 GB
[16:36:58] Train: 25819 cặp, Val: 4300 cặp
[16:36:59] Tách token (một lần)...
[16:37:00]   tách token train: 2000/25819
[16:37:01]   tách token train: 4000/25819
[16:37:02]   tách token train: 6000/25819
[16:37:04]   tách token train: 8000/25819
[16:37:05]   tách token train: 10000/25819
[16:37:06]   tách token train: 12000/25819
[16:37:07]   tách token train: 14000/25819
[16:37:07]   tách token train: 16000/25819
[16:37:09]   tách token train: 18000/25819
[16:37:10]   tách token train: 20000/25819
[16:37:11]   tách token train: 22000/25819
[16:37:12]   tách token train: 24000/25819
[16:37:13]   tách token train: 25819/25819
[16:37:14]   tách token val: 2000/4300
[16:37:15]   tác

In [ ]:
!cp -r results/logs "$DRIVE_OUT"/logs_v6
!ls -la "$DRIVE_OUT/cross_encoder_best"
# Tải thư mục cross_encoder_best từ Drive về máy: đặt vào checkpoints/cross_encoder_best (V8 đọc từ đó)

total 529359
drwx------ 2 root root      4096 Nov 24 16:49 .
drwx------ 6 root root      4096 Nov 24 17:01 ..
-rw------- 1 root root        22 Nov 24 16:49 added_tokens.json
-rw------- 1 root root   1135173 Nov 24 16:49 bpe.codes
-rw------- 1 root root       746 Nov 24 16:49 config.json
-rw------- 1 root root      5189 Nov 24 16:49 linear_head.pt
-rw------- 1 root root 540015464 Nov 24 16:49 model.safetensors
-rw------- 1 root root      1205 Nov 24 16:49 tokenizer_config.json
-rw------- 1 root root    895321 Nov 24 16:49 vocab.txt
